## Example 1 — installed version and the formats libsndfile offers

In [ ]:
import importlib.metadata
import soundfile

assert soundfile.__version__ == importlib.metadata.version("soundfile"), soundfile.__version__
formats = soundfile.available_formats()
assert {"WAV", "FLAC", "OGG", "MP3"} <= set(formats), sorted(formats)
print("soundfile", soundfile.__version__, "on libsndfile", soundfile.__libsndfile_version__)

## Example 2 — WAV and FLAC round trips are sample-exact

In [ ]:
import io
import numpy as np
import soundfile

rate = 16000
tone = (np.sin(2 * np.pi * 440.0 * np.arange(rate) / rate) * 20000).astype(np.int16)
for file_format, subtype in (("WAV", "PCM_16"), ("FLAC", "PCM_16"), ("WAV", "FLOAT")):
    buffer = io.BytesIO()
    soundfile.write(buffer, tone, rate, format=file_format, subtype=subtype)
    buffer.seek(0)
    read_back, read_rate = soundfile.read(buffer, dtype="int16")
    assert read_rate == rate and np.array_equal(read_back, tone), (file_format, subtype)
    print(f"{file_format}/{subtype}: {len(buffer.getvalue())} bytes, {len(read_back)} samples identical")

## Example 3 — the lossy codecs: Vorbis, Opus and MP3 bring the tone back at 440 Hz

In [ ]:
import io
import numpy as np
import soundfile

rate = 48000
tone = (0.5 * np.sin(2 * np.pi * 440.0 * np.arange(rate) / rate)).astype(np.float32)
for file_format, subtype in (("OGG", "VORBIS"), ("OGG", "OPUS"), ("MP3", "MPEG_LAYER_III")):
    assert subtype in soundfile.available_subtypes(file_format), (file_format, subtype)
    buffer = io.BytesIO()
    soundfile.write(buffer, tone, rate, format=file_format, subtype=subtype)
    buffer.seek(0)
    decoded, decoded_rate = soundfile.read(buffer, dtype="float32")
    spectrum = np.abs(np.fft.rfft(decoded))
    peak_hz = float(np.argmax(spectrum) * decoded_rate / len(decoded))
    assert decoded_rate == rate and abs(peak_hz - 440.0) < 2.0, (file_format, subtype, decoded_rate, peak_hz)
    print(f"{file_format}/{subtype}: {len(buffer.getvalue())} bytes, loudest frequency {peak_hz:.1f} Hz")

## Example 4 — `SoundFile` objects: header fields, seeking and reading in blocks

In [ ]:
import io
import numpy as np
import soundfile

rate = 22050
stereo = np.stack([np.linspace(-0.5, 0.5, rate), np.linspace(0.5, -0.5, rate)], axis=1).astype(np.float32)
buffer = io.BytesIO()
soundfile.write(buffer, stereo, rate, format="FLAC", subtype="PCM_24")
buffer.seek(0)
info = soundfile.info(buffer)
assert (info.samplerate, info.channels, info.frames, info.format, info.subtype) == (rate, 2, rate, "FLAC", "PCM_24")
buffer.seek(0)
with soundfile.SoundFile(buffer) as handle:
    assert handle.seek(1000) == 1000
    first = handle.read(10, dtype="float32")
    assert np.allclose(first, stereo[1000:1010], atol=1e-6)
    handle.seek(0)
    total = sum(len(block) for block in handle.blocks(blocksize=4096))
assert total == rate, total
print("FLAC/PCM_24 stereo:", info.frames, "frames,", f"{info.duration:.1f} s; seek and block reads agree")

## Example 5 — bytes that are not audio are refused with soundfile's own error

In [ ]:
import io
import soundfile

try:
    soundfile.read(io.BytesIO(b"this is not an audio file" * 40))
    raise AssertionError("garbage was read as audio")
except soundfile.LibsndfileError as error:
    print("refused:", error)
try:
    soundfile.write(io.BytesIO(), [0.0, 0.1], 16000, format="NOPE")
    raise AssertionError("an unknown format was accepted")
except ValueError as error:
    print("refused:", error)